In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/next-word-predictor-text-generator-dataset/next_word_predictor.txt


In [2]:
with open('/kaggle/input/next-word-predictor-text-generator-dataset/next_word_predictor.txt','r' ,encoding = 'utf-8') as file:

    data = file.read()

In [3]:
def seperate_punc(text):
    return [token.lower() for token in data.split(" ") if token not in '\n\n \n\n\n!"-#$%&()--.*+,-/:;<=>?@[\\]^_`{|}~\t\n ']

In [4]:
tokens = seperate_punc(data)

In [5]:
cleaned_text = ' '.join(tokens)

In [6]:
from tensorflow.keras.preprocessing.text import Tokenizer

2025-11-17 14:44:30.820267: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763390671.034800      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763390671.093003      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [7]:
tokenizer = Tokenizer()

tokenizer.fit_on_texts([cleaned_text])

In [8]:
print(f'total length of the word index: {len(tokenizer.word_index)}')

total length of the word index: 4993


In [9]:
input_sequences = []

for sentence in cleaned_text.split('\n'):

    tokenized_sentence = tokenizer.texts_to_sequences([sentence])[0]

    for i in range(1,len(tokenized_sentence)):
        input_sequences.append(tokenized_sentence[:i+1])



In [10]:
max(len(x) for x in input_sequences)

325

In [11]:
from tensorflow.keras.utils import pad_sequences

In [12]:
padded_input_sequences = pad_sequences(input_sequences, maxlen=325, padding='pre')

In [13]:
X = padded_input_sequences[:,: -1]
y = padded_input_sequences[:,-1]

In [14]:
print(X.shape,y.shape)

(26383, 324) (26383,)


In [15]:
from tensorflow.keras.utils import to_categorical

In [16]:
y = to_categorical(y, num_classes = len(tokenizer.word_index)+1)

In [17]:
y.shape

(26383, 4994)

In [18]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM

In [19]:
model = Sequential()
model.add(Embedding(4994,100, input_length = 324))
model.add(LSTM(150))
model.add(Dense(4994, activation = 'softmax'))

model.compile(loss = 'categorical_crossentropy', optimizer = 'adam', metrics = ['accuracy'])
model.build(input_shape = (None,324))
model.summary()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
I0000 00:00:1763390685.051000      48 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 324, 100)       │       499,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 150)            │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4994)           │       754,094 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,404,094 (5.36 MB)

 Trainable params: 1,404,094 (5.36 MB)

 Non-trainable params: 0 (0.00 B)

In [20]:
model.fit(X,y,epochs = 50)

Epoch 1/50


I0000 00:00:1763390690.492232     105 cuda_dnn.cc:529] Loaded cuDNN version 90300


825/825 ━━━━━━━━━━━━━━━━━━━━ 21s 22ms/step - accuracy: 0.0511 - loss: 7.2870
Epoch 2/50
825/825 ━━━━━━━━━━━━━━━━━━━━ 18s 22ms/step - accuracy: 0.0728 - loss: 6.4227
Epoch 3/50
825/825 ━━━━━━━━━━━━━━━━━━━━ 18s 22ms/step - accuracy: 0.0945 - loss: 5.9499
Epoch 4/50
825/825 ━━━━━━━━━━━━━━━━━━━━ 18s 22ms/step - accuracy: 0.1143 - loss: 5.5736
Epoch 5/50
825/825 ━━━━━━━━━━━━━━━━━━━━ 18s 22ms/step - accuracy: 0.1373 - loss: 5.1785
Epoch 6/50
825/825 ━━━━━━━━━━━━━━━━━━━━ 18s 22ms/step - accuracy: 0.1553 - loss: 4.8168
Epoch 7/50
825/825 ━━━━━━━━━━━━━━━━━━━━ 18s 22ms/step - accuracy: 0.1773 - loss: 4.4719
Epoch 8/50
825/825 ━━━━━━━━━━━━━━━━━━━━ 18s 22ms/step - accuracy: 0.2098 - loss: 4.1386
Epoch 9/50
825/825 ━━━━━━━━━━━━━━━━━━━━ 18s 22ms/step - accuracy: 0.2510 - loss: 3.8271
Epoch 10/50
825/825 ━━━━━━━━━━━━━━━━━━━━ 18s 22ms/step - accuracy: 0.3038 - loss: 3.4989
Epoch 11/50
825/825 ━━━━━━━━━━━━━━━━━━━━ 18s 22ms/step - accuracy: 0.3520 - loss: 3.2266
Epoch 12/50
825/825 ━━━━━━━━━━━━━━━━━━━━ 

In [24]:
import time

In [27]:
text = "excellent"

for i in range(15):
    token_text = tokenizer.texts_to_sequences([text])[0]
    padded_token_text = pad_sequences([token_text], maxlen=324, padding='pre')
    pos = np.argmax(model.predict(padded_token_text))

    for word, index in tokenizer.word_index.items():
        if index == pos:
            text = text + " " + word
            print(text)
            time.sleep(1)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
excellent vendors
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
excellent vendors sold
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
excellent vendors sold a
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
excellent vendors sold a variety
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
excellent vendors sold a variety of
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
excellent vendors sold a variety of goods
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
excellent vendors sold a variety of goods from
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
excellent vendors sold a variety of goods from sizzling
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
excellent vendors sold a variety of goods from sizzling hot
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
excellent vendors sold a variety of goods from sizzling hot dogs
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
excellent vendors sold a variety of goods from sizzling hot dogs to
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
excellent vendors sold a variety of goods from sizzling hot dogs to han

In [28]:
import pickle

In [32]:
model.save('my_model.keras')

In [33]:
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)